In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score


# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
df = pd.read_csv('Q3_data.csv')

In [ ]:
# Task 2: Write your code here:
# Show first rows just to understand what the data looks like
df.head()

In [ ]:
# Task 3: Write your code here:
# Check column names, data types, and missing values
df.info()

In [ ]:
# Task 4: Write your code here:
# See statistics like mean, min, max for numerical columns
df.describe()

In [ ]:
# Task 1: Write your code here:
# Get all categorical columns
cat_cols = df.select_dtypes(include='object').columns

# Replace missing categorical values with 'unknown' so encoding works
df[cat_cols] = df[cat_cols].fillna("unknown")
# Get all numerical columns
num_cols = df.select_dtypes(include='number').columns

# Fill missing numerical values with the mean so the model doesn’t crash
df[num_cols] = df[num_cols].fillna(df[num_cols].mean())

In [ ]:
# Task 2: Write your code here:
df.drop_duplicates(inplace=True)
df.fillna(df.median(numeric_only=True), inplace=True)

In [ ]:
# Task 3: Write your code here:
# Encode Categorical Features
le = LabelEncoder()
for col in df.select_dtypes(include=['object']).columns:
    df[col] = le.fit_transform(df[col])

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X = pd.DataFrame(X_scaled, columns=X.columns)

# IMPORTANT: we NEVER scale y

In [ ]:
# Task 5: Write your code here:
print("Target Distribution:\n", df['target'].value_counts(normalize=True))
# Note: State if imbalanced based on output (e.g., "The target is imbalanced").


In [ ]:
# Task 1: Write your code here:
X = df.drop('target', axis=1)
y = df['target']

In [ ]:
# Task 2,3,4,5: Write your code here:
# 2. we will use here StratifiedKFold since(classification task)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = []

for train_idx, test_idx in skf.split(X, y):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
# 3. # Import a classification model
from sklearn.linear_model import LogisticRegression

# Create the model
model = LogisticRegression()

# Train the CatBoostClassifier model
model = CatBoostClassifier(verbose=0, random_state=42)
model.fit(X_train, y_train)
    # Predict class labels
preds = model.predict(X_test)
scores.append(f1_score(y_test, preds))

print(f"Average F1-Score: {np.mean(scores):.4f}")


In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt
import pandas as pd

# 1. Get feature importance from the trained CatBoost model
importances = model.get_feature_importance()
feature_names = X.columns

# Create a DataFrame for easier plotting
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=True) # Sort for horizontal bar plot

# 2. Plot Feature Importance
plt.figure(figsize=(10, 8))
plt.barh(feature_importance_df['Feature'], feature_importance_df['Importance'], color='gold')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.title('Feature Importance: Finding the Golden Feature')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

# 3. Identifying and printing the 'Golden Feature' :)
golden_feature = feature_importance_df.iloc[-1]['Feature']
golden_score = feature_importance_df.iloc[-1]['Importance']

print(f"The Golden Feature is: {golden_feature}")
print(f"Importance Score: {golden_score:.2f}")

In [ ]:
# Task 2: Write your code here:
golden_feature = feature_importance_df.iloc[-1]['Feature']
golden_score = feature_importance_df.iloc[-1]['Importance']

print(f"The Golden Feature is: {golden_feature}")
print(f"Importance Score: {golden_score:.2f}")

In [ ]:
# Task Bonus: Write your code here:


# 1. Create new X with only the golden featue
golden_feature = 'P_2'
X_golden = X[[golden_feature]]  # Ensure X is DataFrame
y = df['target']

# 2. Run the same StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
golden_model_scores = []

for train_index, test_index in skf.split(X_golden, y):
    X_train, X_test = X_golden.iloc[train_index], X_golden.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Initialize and train CatBoost on single feature p_2
    model_golden = CatBoostClassifier(verbose=0, random_state=42)
    model_golden.fit(X_train, y_train)

    # Evaluate using Accuracy
    preds = model_golden.predict(X_test)
    acc = accuracy_score(y_test, preds)
    golden_model_scores.append(acc)

# 3. Print and compare the accuracy
avg_golden_accuracy = np.mean(golden_model_scores)
print(f"Full Model Accuracy: {avg_full_accuracy:.4f}") # Assume avg_full_accuracy was stored from Part 2
print(f"Golden Feature Only Accuracy: {avg_golden_accuracy:.4f}")
print(f"Performance Retained: {(avg_golden_accuracy / avg_full_accuracy) * 100:.2f}%")